In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('arizona_sunrise_sunset.csv')

def to_utc_minutes(series):
    dt = pd.to_datetime(series, format='ISO8601', utc=True)
    return dt.dt.hour * 60 + dt.dt.minute + dt.dt.second / 60

df['naut_begin_utc_min'] = to_utc_minutes(df['nautical_twilight_begin'])
df['naut_end_utc_min']   = to_utc_minutes(df['nautical_twilight_end'])

# Target windows in UTC minutes since midnight
# Dawn:  11:50–11:59 UTC  (= 04:50–04:59 Arizona local)
# Dusk:  02:59–03:06 UTC  (= 19:59–20:06 Arizona local)
DAWN_LO, DAWN_HI = 11 * 60 + 50, 11 * 60 + 59
DUSK_LO, DUSK_HI =  2 * 60 + 59,  3 * 60 +  6

def range_diff(val, lo, hi):
    if val < lo:   return val - lo
    elif val > hi: return val - hi
    return 0.0

df['dawn_diff'] = df['naut_begin_utc_min'].apply(lambda v: range_diff(v, DAWN_LO, DAWN_HI))
df['dusk_diff'] = df['naut_end_utc_min'].apply(lambda v: range_diff(v, DUSK_LO, DUSK_HI))

df['point'] = df['lat'].round(2).astype(str) + ',' + df['lon'].round(2).astype(str)

heatmap_data = df.set_index('point')[['dawn_diff', 'dusk_diff']].T
heatmap_data.index = [
    'Nautical Dawn vs 11:50–11:59 UTC',
    'Nautical Dusk vs 02:59–03:06 UTC',
]

plt.figure(figsize=(20, 3))
sns.heatmap(
    heatmap_data,
    cmap='RdYlGn_r',
    center=0,
    annot=True,
    fmt='.1f',
    cbar_kws={'label': 'Minutes outside window (0 = within target)'},
    linewidths=0.3,
)
plt.title('Nautical Twilight deviation from target UTC windows across Arizona')
plt.xlabel('lat, lon')
plt.tight_layout()
plt.show()


In [ ]:
import folium
import branca.colormap as cm

STEP = 0.5

clat = (df['lat'].min() + df['lat'].max()) / 2
clon = (df['lon'].min() + df['lon'].max()) / 2
m = folium.Map(location=[clat, clon], zoom_start=6, tiles='OpenStreetMap')

def make_layer(col, label, caption, window_str):
    vmax = df[col].abs().max()
    cmap = cm.LinearColormap(
        ['#2ecc71', '#f9f9f9', '#e74c3c'],
        vmin=-vmax, vmax=vmax,
        caption=caption,
    )
    layer = folium.FeatureGroup(name=label, show=(col == 'dawn_diff'))
    for _, row in df.iterrows():
        diff = row[col]
        note = f"{abs(diff):.1f} min early" if diff < 0 else (f"{diff:.1f} min late" if diff > 0 else "within window")
        folium.Rectangle(
            bounds=[
                [row['lat'] - STEP / 2, row['lon'] - STEP / 2],
                [row['lat'] + STEP / 2, row['lon'] + STEP / 2],
            ],
            color=None,
            fill=True,
            fill_color=cmap(diff),
            fill_opacity=0.65,
            tooltip=(
                f"<b>lat {row['lat']}, lon {row['lon']}</b><br>"
                f"Target: {window_str} UTC<br>"
                f"{note}"
            ),
        ).add_to(layer)
    return layer, cmap

dawn_layer, dawn_cmap = make_layer(
    'dawn_diff', 'Nautical Dawn (11:50–11:59 UTC)', 'Dawn diff (min)', '11:50–11:59'
)
dusk_layer, dusk_cmap = make_layer(
    'dusk_diff', 'Nautical Dusk (02:59–03:06 UTC)', 'Dusk diff (min)', '02:59–03:06'
)

dawn_layer.add_to(m)
dusk_layer.add_to(m)
dawn_cmap.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

m.save('arizona_sunrise_heatmap.html')
m


In [ ]:
import folium
import branca.colormap as cm

# Combined score: total minutes outside both windows (0 = both on target)
df['combined'] = df['dawn_diff'].abs() + df['dusk_diff'].abs()

STEP = 0.5
clat = (df['lat'].min() + df['lat'].max()) / 2
clon = (df['lon'].min() + df['lon'].max()) / 2
m = folium.Map(location=[clat, clon], zoom_start=6, tiles='OpenStreetMap')

# ── Individual layers ─────────────────────────────────────────────────────────
def make_layer(col, label, window_str, show):
    vmax = df[col].abs().max()
    cmap = cm.LinearColormap(
        ['#2ecc71', '#f9f9f9', '#e74c3c'],
        vmin=-vmax, vmax=vmax,
    )
    layer = folium.FeatureGroup(name=label, show=show)
    for _, row in df.iterrows():
        diff = row[col]
        note = f"{abs(diff):.1f} min early" if diff < 0 else (f"{diff:.1f} min late" if diff > 0 else "within window")
        folium.Rectangle(
            bounds=[[row['lat']-STEP/2, row['lon']-STEP/2],
                    [row['lat']+STEP/2, row['lon']+STEP/2]],
            color=None, fill=True,
            fill_color=cmap(diff), fill_opacity=0.65,
            tooltip=f"<b>{row['lat']}, {row['lon']}</b><br>Target: {window_str} UTC<br>{note}",
        ).add_to(layer)
    return layer

# ── Overlap layer ─────────────────────────────────────────────────────────────
vmax_combined = df['combined'].max()
overlap_cmap = cm.LinearColormap(
    ['#27ae60', '#f1c40f', '#e74c3c'],
    vmin=0, vmax=vmax_combined,
    caption='Total deviation dawn + dusk (min)  |  green = both on target',
)
overlap_layer = folium.FeatureGroup(name='Overlap (dawn + dusk combined)', show=True)
for _, row in df.iterrows():
    dawn_note = (f"Dawn: {abs(row['dawn_diff']):.1f} min early" if row['dawn_diff'] < 0
                 else f"Dawn: {row['dawn_diff']:.1f} min late" if row['dawn_diff'] > 0
                 else "Dawn: within window")
    dusk_note = (f"Dusk: {abs(row['dusk_diff']):.1f} min early" if row['dusk_diff'] < 0
                 else f"Dusk: {row['dusk_diff']:.1f} min late" if row['dusk_diff'] > 0
                 else "Dusk: within window")
    folium.Rectangle(
        bounds=[[row['lat']-STEP/2, row['lon']-STEP/2],
                [row['lat']+STEP/2, row['lon']+STEP/2]],
        color=None, fill=True,
        fill_color=overlap_cmap(row['combined']), fill_opacity=0.70,
        tooltip=(f"<b>{row['lat']}, {row['lon']}</b><br>"
                 f"{dawn_note}<br>{dusk_note}<br>"
                 f"<b>Total: {row['combined']:.1f} min off</b>"),
    ).add_to(overlap_layer)

make_layer('dawn_diff', 'Dawn only (11:50–11:59 UTC)', '11:50–11:59', show=False).add_to(m)
make_layer('dusk_diff', 'Dusk only (02:59–03:06 UTC)', '02:59–03:06', show=False).add_to(m)
overlap_layer.add_to(m)
overlap_cmap.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

m.save('arizona_sunrise_heatmap.html')
m
